In [1]:
import folium
import geopandas as gpd
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.features import rasterize
from scipy import ndimage

In [2]:
def calculate_flood_area_local(dem_path, rhein_path, output_path, rise=1.0):
    rhein = gpd.read_file(rhein_path)

    with rasterio.open(dem_path) as src:
        dem = src.read(1)
        profile = src.profile.copy()
        nodata = src.nodata
        transform = src.transform
        crs = src.crs

    rhein = rhein.to_crs(crs)

    valid = dem != nodata if nodata is not None else np.ones_like(dem, dtype=bool)

    # Rheinfläche ins Raster brennen
    rhein_mask = rasterize(
        [(geom, 1) for geom in rhein.geometry],
        out_shape=dem.shape,
        transform=transform,
        fill=0,
        dtype="uint8"
    ).astype(bool)

    # Für jedes Pixel: Index des nächstgelegenen Rhein-Pixels
    _, indices = ndimage.distance_transform_edt(
        ~rhein_mask,
        return_indices=True
    )

    nearest_rhein_rows = indices[0]
    nearest_rhein_cols = indices[1]

    # Höhe des nächstgelegenen Rhein-Pixels
    local_rhein_height = dem[nearest_rhein_rows, nearest_rhein_cols]

    # Lokaler Wasserstand = Rheinpixel + 1 m
    local_water_level = local_rhein_height + rise

    # Bathtub-Maske
    potential_flood = (dem <= local_water_level) & valid

    # Nur Flächen behalten, die mit dem Rhein verbunden sind
    labels, _ = ndimage.label(
        potential_flood,
        structure=np.ones((3, 3))
    )

    rhein_labels = np.unique(labels[rhein_mask])
    rhein_labels = rhein_labels[rhein_labels != 0]

    flood = np.isin(labels, rhein_labels)

    # Neues Höhenmodell erstellen
    new_dem = dem.copy().astype("float32")

    # Nur überflutete Pixel auf lokalen Wasserstand setzen
    new_dem[flood] = local_water_level[flood]

    # NoData behandeln
    if nodata is not None:
        new_dem[~valid] = nodata
    else:
        nodata = -9999
        new_dem[~valid] = nodata

    # Wichtig bei VRT als Input
    profile.update({
        "driver": "GTiff",
        "dtype": "float32",
        "count": 1,
        "nodata": nodata,
        "height": new_dem.shape[0],
        "width": new_dem.shape[1],
        "transform": transform,
        "compress": "lzw"
    })

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(new_dem, 1)

    print(f"Überflutungsmodell erstellt: {output_path}")

In [3]:
calculate_flood_area_local(
    dem_path="../Data/Hoehenmodell/Hoehenmodell.vrt",
    rhein_path="../Data/Rhein/rhein.gpkg",
    output_path="../Data/Hoehenmodell/Berechnung/rhein_plus10m_lokal.tif",
    rise=10.0
)

c:\Users\jasmi\miniconda3\envs\geopython\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'rhein.gpkg': 'Rhein' (default), 'Rhein_flaeche_abs'. Specify layer parameter to avoid this warning.
  result = read_func(


Überflutungsmodell erstellt: ../Data/Hoehenmodell/Berechnung/rhein_plus10m_lokal.tif


In [ ]:
# Dateien
vrt_pfad = "../Data/Hoehenmodell/Hoehenmodell.vrt"
neu_pfad = "../Data/Hoehenmodell/Berechnung/rhein_plus10m_lokal.tif" #Parameter anpassen für die anderen Höhen
ausgabe_pfad = "../Data/Hoehenmodell/Berechnung/veraenderte_hoehen.tif"

# Original DEM laden
with rasterio.open(vrt_pfad) as src_alt:

    alt = src_alt.read(1)

    meta = src_alt.meta.copy()

    original_nodata = src_alt.nodata

# Neues DEM laden
with rasterio.open(neu_pfad) as src_neu:

    neu = src_neu.read(1)

# Falls kein NoData definiert ist
nodata = original_nodata if original_nodata is not None else -9999

# Leeres Raster erstellen
ergebnis = np.full(
    alt.shape,
    nodata,
    dtype=neu.dtype
)

# Geänderte Pixel finden
maske = (
    (np.abs(neu - alt) > 0.01) &
    (alt != nodata) &
    (neu != nodata)
)

# Nur neue Werte übernehmen
ergebnis[maske] = neu[maske]

# Meta für TIFF anpassen
meta.update(
    driver="GTiff",
    nodata=nodata,
    dtype=ergebnis.dtype
)

# Neues Raster schreiben
with rasterio.open(
    ausgabe_pfad,
    "w",
    **meta
) as dst:

    dst.write(ergebnis, 1)

print("Fertig:", ausgabe_pfad)

Fertig: ../Data/Hoehenmodell/Berechnung/veraenderte_hoehen.tif
